# Student-selected 50:50 LightGBM leaf-size blend

**12 October 2026:** the student explicitly chose an equal blend of the 5-example and 50-example LightGBM versions. Both use 150 trees, all 165 original features, confirmed labels and the same previously chosen settings. The only new modeling choice is **0.5 × leaf-5 probability + 0.5 × leaf-50 probability**. No weights are searched or adjusted after viewing results.

Leaf 50 improved the all-15, later-seven and four-check means, but declined on the latest check and worst check. Averaging tests whether combining the different prediction rules improves ranking. It does not guarantee improvement or make the weaker periods disappear.

Use the same 15 chronological development checks and the same two component fits at each cutoff. An optional verified cache reuses the already measured predictions and models from this student's leaf-size experiment, avoiding duplicate training. If it is unavailable or fails its checks, this notebook can fit both components and the unchanged ensemble validation control from the supplied data alone. The generated prediction CSV requires only this notebook, supplied competition files and recorded libraries; prior outputs are optional.

The four reference views are the reproduced submitted ensemble recipe, leaf 5, leaf 50 and the fixed blend. No shorter-training experiment, leaf 20 refinement, semi-supervised mechanism or other blend is added. Unknown rows do not receive labels. Future checking and test rows do not fit a component or preprocessing step. The checks overlap and have been reused for development; local AUC does not establish a future Kaggle score. The current verified public score is 0.94701, and all standalone/blended LightGBM public scores remain unknown.

**AI assistance:** the student chose this fixed blend after reviewing the leaf-size tradeoffs. OpenAI Codex implemented and evaluated the selected combination, verified predictions and drafted this factual record. The student owns the interpretation and final submission decision and must disclose assistance. No Kaggle upload occurs.

Run all cells in a fresh CPU kernel. Set DATA_DIR, OUTPUT_DIR and optional CACHE_DIR if needed. Components are trained using only provided competition train/test/sample files and recorded libraries.


In [1]:
from pathlib import Path
import os, json, hashlib, time, platform
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "4"
import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, Booster
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from threadpoolctl import threadpool_limits

DATA_DIR = Path(os.environ.get("BT4012_DATA_DIR", "/Users/mlvlo/BT4012/Kaggle/Data Set"))
if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT_DIR = Path(os.environ.get("BT4012_LEAF_BLEND_OUTPUT_DIR", str(Path.cwd() / "lightgbm_leaf_blend_results")))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = [f"feat_{i}" for i in range(1, 166)]
LEAF_SIZES = (5, 50)
CHECKS = tuple((end, end + 1, end + 14) for end in range(7, 22))
SELECTED_FOUR = (14, 17, 19, 21)
SEED = 42
STARTED = time.perf_counter()
FIXED_LGBM = dict(n_estimators=150, learning_rate=0.05, num_leaves=15, max_bin=63,
                 min_child_weight=0.001, reg_lambda=1,
                 objective="binary", deterministic=True, force_col_wise=True,
                 verbosity=-1, n_jobs=4, random_state=SEED)
RECIPE = dict(min_child_samples_values=list(LEAF_SIZES), fixed_blend_weights={"leaf5": 0.5, "leaf50": 0.5}, fixed_lightgbm=FIXED_LGBM,
              chronological_checks=[list(check) for check in CHECKS],
              selected_four=list(SELECTED_FOUR), existing_public_auc=0.94701,
              student_decision="Test the fixed 5/50 blend (Recommended)",
              final_selection="pending student review", kaggle_upload=False)
recipe_file = OUTPUT_DIR / "declared_recipe.json"
if recipe_file.exists():
    assert json.loads(recipe_file.read_text()) == RECIPE, "Different prior recipe: use a new output folder."
else:
    recipe_file.write_text(json.dumps(RECIPE, indent=2))

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def provided_file(name):
    matches = sorted(DATA_DIR.rglob(name))
    assert len(matches) == 1, f"Expected exactly one provided {name}: found {matches}"
    return matches[0]

def progress(phase, **details):
    (OUTPUT_DIR / "progress.json").write_text(json.dumps(dict(
        phase=phase, elapsed_seconds=round(time.perf_counter() - STARTED, 3), **details), indent=2))

print("Declared minimum leaf sizes:", LEAF_SIZES, "; fixed settings:", FIXED_LGBM)
print("Results folder:", OUTPUT_DIR)

# Optional cache of this student's already measured component fits. If absent,
# every prediction below can be reproduced by fitting only the provided data.
CACHE_DIR = Path(os.environ.get("BT4012_LEAF_CACHE_DIR", "/Users/mlvlo/BT4012/Kaggle/lightgbm_leaf_results"))

Declared minimum leaf sizes: (5, 50) ; fixed settings: {'n_estimators': 150, 'learning_rate': 0.05, 'num_leaves': 15, 'max_bin': 63, 'min_child_weight': 0.001, 'reg_lambda': 1, 'objective': 'binary', 'deterministic': True, 'force_col_wise': True, 'verbosity': -1, 'n_jobs': 4, 'random_state': 42}
Results folder: /Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/lightgbm_leaf_blend_results


In [2]:
train_path = provided_file("Train Data.csv")
test_path = provided_file("Test Data.csv")
sample_path = provided_file("BT4012 Competition Submission.csv")
SOURCE_HASHES = {str(p): digest(p) for p in (train_path, test_path, sample_path)}
train = pd.read_csv(train_path)
numeric_labels = pd.to_numeric(train["label"], errors="coerce")
known = train.loc[numeric_labels.notna()].copy()
known["label"] = numeric_labels.loc[numeric_labels.notna()].astype(int)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)
assert numeric_labels.dropna().isin([0, 1]).all()
assert known["label"].nunique() == 2
assert train["txId"].is_unique and test["txId"].is_unique
assert set(train["txId"]).isdisjoint(test["txId"])
assert train["time_step"].max() < test["time_step"].min()
assert test["index"].is_unique and sample["index"].is_unique
assert set(test["index"]) == set(sample["index"])
assert sample.columns.tolist() == ["index", "target"]
for rows in (train, test):
    assert np.isfinite(rows[FEATURES].to_numpy()).all()

def make_lightgbm(minimum):
    assert minimum in LEAF_SIZES
    return LGBMClassifier(min_child_samples=minimum, **FIXED_LGBM)

def existing_xgb_columns(learning):
    # Preserve the submitted ensemble's filter, using fitting data only.
    cutoff = int(learning["time_step"].max())
    filter_rows = learning.loc[learning["time_step"] <= cutoff - 3]
    correlations = filter_rows[FEATURES].corrwith(filter_rows["time_step"], method="spearman")
    removed = correlations.index[correlations.abs() > 0.9].tolist()
    return [column for column in FEATURES if column not in removed], removed

def make_existing_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.05, tree_method="hist",
                         grow_policy="lossguide", max_depth=0, max_leaves=15,
                         max_bin=127, min_child_weight=1, reg_lambda=1,
                         objective="binary:logistic", n_jobs=4, random_state=SEED)

print(f"Training rows: {len(train):,}; confirmed labels: {len(known):,}; "
      f"unknown rows: {int(numeric_labels.isna().sum()):,}; test rows: {len(test):,}")
print("Confirmed labels:", known["label"].value_counts().to_dict())

cache_files = list(CACHE_DIR.glob('*')) if CACHE_DIR.exists() else []
cache_hashes = {str(p): digest(p) for p in cache_files if p.is_file()}
cache_meta_path = CACHE_DIR / 'experiment_metadata.json'
cache_valid = False
if cache_meta_path.exists():
    cache_meta = json.loads(cache_meta_path.read_text())
    cache_valid = (cache_meta.get('fixed_lightgbm') == FIXED_LGBM
                   and cache_meta.get('min_child_samples_values') == [5, 20, 50]
                   and sorted(cache_meta.get('source_sha256', {}).values()) == sorted(SOURCE_HASHES.values())
                   and cache_meta.get('lightgbm') == lightgbm.__version__
                   and cache_meta.get('xgboost') == xgboost.__version__
                   and cache_meta.get('all_saved_models_verified') is True
                   and cache_meta.get('future_rows_used_for_fitting') is False
                   and cache_meta.get('outer_auc_used_for_stopping') is False)
print('Verified optional component cache available:', cache_valid)

Training rows: 141,772; confirmed labels: 31,235; unknown rows: 110,537; test rows: 15,329
Confirmed labels: {0: 27591, 1: 3644}
Verified optional component cache available: True


In [3]:
VARIANTS = ('current_pair', 'lightgbm_leaf_5', 'lightgbm_leaf_50', 'lightgbm_leaf_5_50_blend')
arrays, records = {}, []
cached_arrays = {}
if cache_valid and (CACHE_DIR / 'validation_probabilities.npz').exists():
    with np.load(CACHE_DIR / 'validation_probabilities.npz') as saved:
        cached_arrays = {key: saved[key] for key in saved.files}
cache_checks_used = 0
for cutoff, first, last in CHECKS:
    progress('fixed blend checking', cutoff=cutoff)
    learning = known.loc[known.time_step <= cutoff]
    checking = known.loc[known.time_step.between(first, last)]
    assert learning.time_step.max() < checking.time_step.min()
    assert set(learning.txId).isdisjoint(checking.txId)
    labels = checking.label.to_numpy()
    cache_keys = [f'{name}_{cutoff}' for name in VARIANTS[:3]] + [f'labels_{cutoff}', f'txIds_{cutoff}']
    use_cache = all(key in cached_arrays for key in cache_keys)
    if use_cache:
        assert np.array_equal(labels, cached_arrays[f'labels_{cutoff}'])
        assert np.array_equal(checking.txId.to_numpy(), cached_arrays[f'txIds_{cutoff}'])
        probabilities = {name: np.asarray(cached_arrays[f'{name}_{cutoff}'], dtype=np.float64) for name in VARIANTS[:3]}
        cache_checks_used += 1
    else:
        probabilities = {}
        with threadpool_limits(limits=4):
            for minimum in LEAF_SIZES:
                model = make_lightgbm(minimum)
                model.fit(learning[FEATURES], learning.label)
                probabilities[f'lightgbm_leaf_{minimum}'] = model.predict_proba(checking[FEATURES])[:, 1]
            columns, _ = existing_xgb_columns(learning)
            control = make_existing_xgb()
            control.fit(learning[columns], learning.label)
            p_xgb = control.predict_proba(checking[columns])[:, 1]
        probabilities['current_pair'] = 0.5 * p_xgb + 0.5 * probabilities['lightgbm_leaf_5']
    probabilities['lightgbm_leaf_5_50_blend'] = 0.5 * probabilities['lightgbm_leaf_5'] + 0.5 * probabilities['lightgbm_leaf_50']
    row = dict(train_end=cutoff, validation_start=first, validation_end=last,
               learning_rows=len(learning), checking_rows=len(checking), reused_verified_components=use_cache)
    for name in VARIANTS:
        values = probabilities[name]
        assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()
        row[name + '_auc'] = float(roc_auc_score(labels, values))
        arrays[f'{name}_{cutoff}'] = values
    row['blend_minus_leaf5'] = row['lightgbm_leaf_5_50_blend_auc'] - row['lightgbm_leaf_5_auc']
    row['blend_minus_leaf50'] = row['lightgbm_leaf_5_50_blend_auc'] - row['lightgbm_leaf_50_auc']
    row['blend_minus_pair'] = row['lightgbm_leaf_5_50_blend_auc'] - row['current_pair_auc']
    arrays[f'labels_{cutoff}'] = labels
    arrays[f'txIds_{cutoff}'] = checking.txId.to_numpy()
    arrays[f'time_steps_{cutoff}'] = checking.time_step.to_numpy()
    records.append(row)
    print(f"Cutoff {cutoff}: leaf 5 {row['lightgbm_leaf_5_auc']:.6f}; leaf 50 {row['lightgbm_leaf_50_auc']:.6f}; "
          f"fixed blend {row['lightgbm_leaf_5_50_blend_auc']:.6f}.")
validation = pd.DataFrame(records)
validation.to_csv(OUTPUT_DIR / 'validation_results.csv', index=False)
np.savez_compressed(OUTPUT_DIR / 'validation_probabilities.npz', **arrays)
assert validation.train_end.tolist() == list(range(7, 22))
display(validation[['train_end'] + [name + '_auc' for name in VARIANTS]])

Cutoff 7: leaf 5 0.991691; leaf 50 0.992588; fixed blend 0.992600.
Cutoff 8: leaf 5 0.993200; leaf 50 0.993597; fixed blend 0.993669.
Cutoff 9: leaf 5 0.990934; leaf 50 0.991773; fixed blend 0.991556.
Cutoff 10: leaf 5 0.976330; leaf 50 0.975622; fixed blend 0.976637.
Cutoff 11: leaf 5 0.973871; leaf 50 0.971731; fixed blend 0.973352.
Cutoff 12: leaf 5 0.968787; leaf 50 0.978432; fixed blend 0.975293.
Cutoff 13: leaf 5 0.969207; leaf 50 0.976040; fixed blend 0.974217.
Cutoff 14: leaf 5 0.968360; leaf 50 0.978591; fixed blend 0.975515.
Cutoff 15: leaf 5 0.943194; leaf 50 0.947238; fixed blend 0.945932.
Cutoff 16: leaf 5 0.948616; leaf 50 0.946034; fixed blend 0.947183.
Cutoff 17: leaf 5 0.941784; leaf 50 0.946074; fixed blend 0.943988.
Cutoff 18: leaf 5 0.923267; leaf 50 0.929097; fixed blend 0.925994.
Cutoff 19: leaf 5 0.918555; leaf 50 0.927562; fixed blend 0.923514.
Cutoff 20: leaf 5 0.908792; leaf 50 0.905247; fixed blend 0.907018.
Cutoff 21: leaf 5 0.919396; leaf 50 0.916709; fixed

,train_end,current_pair_auc,lightgbm_leaf_5_auc,lightgbm_leaf_50_auc,lightgbm_leaf_5_50_blend_auc
0,7,0.992985,0.991691,0.992588,0.992600
1,8,0.993827,0.993200,0.993597,0.993669
2,9,0.991915,0.990934,0.991773,0.991556
3,10,0.977749,0.976330,0.975622,0.976637
4,11,0.975593,0.973871,0.971731,0.973352
5,12,0.969104,0.968787,0.978432,0.975293
6,13,0.971758,0.969207,0.976040,0.974217
7,14,0.972177,0.968360,0.978591,0.975515
8,15,0.939024,0.943194,0.947238,0.945932
9,16,0.947183,0.948616,0.946034,0.947183


In [4]:
GROUPS = {'all_15': validation, 'earlier_eight': validation.loc[validation.train_end <= 14],
          'later_seven': validation.loc[validation.train_end >= 15],
          'selected_four': validation.loc[validation.train_end.isin(SELECTED_FOUR)],
          'latest': validation.loc[validation.train_end == 21]}
summary_rows = []
for group, rows in GROUPS.items():
    for name in VARIANTS:
        values = rows[name + '_auc']
        record = dict(group=group, model=name, checks=len(rows), mean_auc=float(values.mean()), worst_auc=float(values.min()))
        for suffix, reference in [('pair', 'current_pair'), ('leaf5', 'lightgbm_leaf_5'), ('leaf50', 'lightgbm_leaf_50')]:
            record['mean_delta_vs_' + suffix] = float((values - rows[reference + '_auc']).mean())
            record['wins_vs_' + suffix] = int((values > rows[reference + '_auc']).sum())
        summary_rows.append(record)
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
per_period = []
for cutoff, _, _ in CHECKS:
    steps, labels = arrays[f'time_steps_{cutoff}'], arrays[f'labels_{cutoff}']
    for step in np.unique(steps):
        mask = steps == step
        record = dict(train_end=cutoff, time_step=int(step), rows=int(mask.sum()), illicit=int(labels[mask].sum()))
        for name in VARIANTS:
            record[name + '_auc'] = float(roc_auc_score(labels[mask], arrays[f'{name}_{cutoff}'][mask])) if len(np.unique(labels[mask])) == 2 else np.nan
        per_period.append(record)
pd.DataFrame(per_period).to_csv(OUTPUT_DIR / 'per_time_step.csv', index=False)
checking_ids = np.concatenate([arrays[f'txIds_{end}'] for end, _, _ in CHECKS])
unique_ids, appearances = np.unique(checking_ids, return_counts=True)
display(summary)
print(f'{cache_checks_used} of 15 checks reused verified component predictions; no weights searched.')

15 of 15 checks reused verified component predictions; no weights searched.


,group,model,checks,mean_auc,worst_auc,mean_delta_vs_pair,wins_vs_pair,mean_delta_vs_leaf5,wins_vs_leaf5,mean_delta_vs_leaf50,wins_vs_leaf50
0,all_15,current_pair,15,0.954972,0.907205,0.000000,0,-0.000761,8,-0.003451,8
1,all_15,lightgbm_leaf_5,15,0.955732,0.908792,0.000761,7,0.000000,0,-0.002690,5
2,all_15,lightgbm_leaf_50,15,0.958422,0.905247,0.003451,7,0.002690,10,0.000000,0
3,all_15,lightgbm_leaf_5_50_blend,15,0.957620,0.907018,0.002649,9,0.001888,11,-0.000802,7
4,earlier_eight,current_pair,8,0.980638,0.969104,0.000000,0,0.001591,8,-0.001658,5
5,earlier_eight,lightgbm_leaf_5,8,0.979047,0.968360,-0.001591,0,0.000000,0,-0.003249,2
6,earlier_eight,lightgbm_leaf_50,8,0.982297,0.971731,0.001658,3,0.003249,6,0.000000,0
7,earlier_eight,lightgbm_leaf_5_50_blend,8,0.981605,0.973352,0.000966,3,0.002558,7,-0.000692,4
8,later_seven,current_pair,7,0.925638,0.907205,0.000000,0,-0.003448,0,-0.005499,3
9,later_seven,lightgbm_leaf_5,7,0.929086,0.908792,0.003448,7,0.000000,0,-0.002051,3


In [5]:
progress('full component verification and fixed blend export')
full_predictions, component_rows = {}, []
cached_final = pd.read_csv(CACHE_DIR / 'final_training.csv') if cache_valid else pd.DataFrame()
for minimum in LEAF_SIZES:
    cached_model_path = CACHE_DIR / f'lightgbm_leaf_{minimum}_model.txt'
    reused = False
    if cache_valid and cached_model_path.exists() and not cached_final.empty:
        cache_row = cached_final.loc[cached_final.min_child_samples == minimum].iloc[0]
        assert digest(cached_model_path) == cache_row.model_sha256
        model = Booster(model_file=str(cached_model_path))
        reused = True
    else:
        classifier = make_lightgbm(minimum)
        with threadpool_limits(limits=4): classifier.fit(known[FEATURES], known.label)
        model = classifier.booster_
    assert model.feature_name() == FEATURES and model.num_trees() == 150
    with threadpool_limits(limits=4): probabilities = np.asarray(model.predict(test[FEATURES], num_threads=4), dtype=np.float64)
    aligned = sample['index'].map(pd.Series(probabilities, index=test['index'])).to_numpy()
    assert np.isfinite(aligned).all() and ((aligned >= 0) & (aligned <= 1)).all()
    if reused:
        with np.load(CACHE_DIR / 'test_probabilities.npz') as saved:
            assert np.array_equal(saved['indices'], sample['index'].to_numpy())
            assert np.allclose(aligned, saved[f'lightgbm_leaf_{minimum}'], rtol=0, atol=1e-15)
    model_path = OUTPUT_DIR / f'lightgbm_leaf_{minimum}_model.txt'
    model.save_model(str(model_path))
    with threadpool_limits(limits=4):
        restored = Booster(model_file=str(model_path)).predict(test[FEATURES], num_threads=4)
    assert np.allclose(probabilities, restored, rtol=0, atol=1e-15)
    full_predictions[f'lightgbm_leaf_{minimum}'] = aligned
    component_rows.append(dict(min_child_samples=minimum, trees=150, training_rows=len(known),
                              reused_verified_model=reused, model_sha256=digest(model_path)))
full_predictions['lightgbm_leaf_5_50_blend'] = 0.5 * full_predictions['lightgbm_leaf_5'] + 0.5 * full_predictions['lightgbm_leaf_50']
full_predictions['indices'] = sample['index'].to_numpy()
csv_path = OUTPUT_DIR / 'submission_lightgbm_leaf_5_50_blend_experimental.csv'
pd.DataFrame({'index': sample['index'], 'target': full_predictions['lightgbm_leaf_5_50_blend']}).to_csv(csv_path, index=False)
reread = pd.read_csv(csv_path)
assert reread.columns.tolist() == ['index', 'target'] and reread['index'].equals(sample['index'])
assert len(reread) == len(test) and np.isfinite(reread.target).all() and reread.target.between(0, 1).all()
assert np.allclose(reread.target, full_predictions['lightgbm_leaf_5_50_blend'], rtol=0, atol=1e-15)
np.savez_compressed(OUTPUT_DIR / 'test_probabilities.npz', **full_predictions)
pd.DataFrame(component_rows).to_csv(OUTPUT_DIR / 'full_components.csv', index=False)
metadata = dict(RECIPE, source_sha256=SOURCE_HASHES, known_training_rows=len(known),
    unknown_training_rows=int(numeric_labels.isna().sum()), test_rows=len(test), cache_validation_checks=cache_checks_used,
    full_models_reused=sum(int(row['reused_verified_model']) for row in component_rows),
    unique_checking_transactions=len(unique_ids), checking_row_appearances=len(checking_ids),
    maximum_checking_appearances=int(appearances.max()), future_rows_used_for_fitting=False,
    unknown_rows_given_labels=False, blend_weights_tuned=False, all_saved_models_verified=True,
    original_data_unchanged=all(digest(path) == value for path, value in SOURCE_HASHES.items()),
    original_cache_unchanged=all(digest(path) == value for path, value in cache_hashes.items()),
    submission_sha256=digest(csv_path), python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
    sklearn=sklearn.__version__, lightgbm=lightgbm.__version__, xgboost=xgboost.__version__,
    elapsed_seconds=round(time.perf_counter() - STARTED, 3))
assert metadata['original_data_unchanged'] and metadata['original_cache_unchanged']
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
progress('complete', chronological_checks=15, fixed_blend_predictions=len(test))
print('Verified fixed equal blend and both component models; no upload or further search.')
print(csv_path)

Verified fixed equal blend and both component models; no upload or further search.
/Users/mlvlo/.codex/.chatgpt-projects/g-p-6a901a23216481918fe5444e40decc24/student_model_review/lightgbm_leaf_blend_results/submission_lightgbm_leaf_5_50_blend_experimental.csv


## Measured findings

The student-selected **50:50 average of the leaf-5 and leaf-50 LightGBM predictions** completed all 15 chronological checks and generated **15,329 verified test predictions**. The initial evaluation reused component predictions and both full models after verifying settings, data hashes, checking identifiers and model hashes; no search over weights was needed. Initial evaluation and export took **3.1 seconds** on CPU. A separate fresh reproduction refitted the same chosen recipe with the optional cache disabled and exactly matched every probability and the final CSV. The components use **31,235 confirmed labels** through step 35; **110,537 unknown rows** received no invented labels.

| Check group | Submitted ensemble recipe | Leaf 5 | Leaf 50 | Fixed 5/50 blend |
|---|---:|---:|---:|---:|
| All 15 | 0.954972 | 0.955732 | 0.958422 | 0.957620 |
| Earlier eight | 0.980638 | 0.979047 | 0.982297 | 0.981605 |
| Later seven | 0.925638 | 0.929086 | 0.931137 | 0.930209 |
| Previously chosen four | 0.935353 | 0.937024 | 0.942234 | 0.940213 |
| Latest: train 1–21, check 22–35 | 0.916956 | 0.919396 | 0.916709 | 0.917834 |

Against the submitted ensemble recipe, the fixed blend improved **6 of seven later checks**, with mean change **+0.004571 AUC**; it improved **4 of the four previously chosen checks**. Against leaf 5, its later-seven mean changed by **+0.001123**, with **4 of seven** wins. Against leaf 50, its later-seven mean changed by **-0.000928**, with **3 of seven** wins. Its latest score changed by **+0.000878** against the submitted ensemble recipe, **-0.001562** against leaf 5 and **+0.001124** against leaf 50.

**AI-assisted recommendation:** The equal blend merits review as a compromise candidate: it improves the later-seven mean over leaf 5 and the submitted ensemble recipe, and improves the latest and worst-check scores over leaf 50. Leaf 50 still has the higher all-15, later-seven and four-check means; leaf 5 still has the higher latest-check and worst-check scores. No candidate dominates every criterion. Preserve these tradeoffs when choosing a submission; hidden-score superiority remains unestablished. All three standalone/blended LightGBM public scores remain unknown. The verified public high score remains **0.94701**. These development results do not establish or forecast a score above 0.96.

### Explanation and tradeoffs

For every transaction, the selected candidate takes the two model scores, adds them and divides by two. This is probability averaging, not an average of AUC values or a vote on binary labels. ROC-AUC is recalculated from the resulting transaction rankings; it can be better or worse than either component's AUC. Both components see the same fitting rows and original features, but their leaf-size constraints create different prediction rules.

The blend improves leaf 50's weakest and latest checks while giving up some average AUC. Its worst fourteen-period AUC is **0.907018**, versus leaf 50 **0.905247**, leaf 5 **0.908792** and the submitted ensemble recipe **0.907205**. The tradeoff cannot be resolved by asserting that a blend is automatically safer or that the highest mean must win on Kaggle.

Weights were fixed at 0.5/0.5 before evaluation and were not revised after examining results. Every component is the same selected recipe recorded in Section 34; no additional feature engineering or label estimation is introduced. All outer windows remain chronological. The **163,794 checking-row appearances cover 21,682 unique transactions**, with up to **14 appearances each. These overlapping, reused development checks support comparisons but not independent held-out confidence or calibrated hidden-score predictions.

### Verification, standalone reproduction and disclosure

All five code cells executed with genuine captured outputs. Independent review recalculated every AUC and summary, verified each checking blend against the original component arrays, and verified the full test blend, identifiers, row count, probability range and CSV round trip. Both copied component models reproduced the original predictions after reload to within 1e−15. Original data and optional component-cache files remained unchanged.

The initial run reused this student's verified outputs. The notebook also contains complete fitting code: if the optional cache is absent, it fits the two components and validation control using the supplied competition files alone. That complete fitting branch was separately executed and verified in **30.1 seconds**, using zero cached validation checks or full models. Every validation/reference probability and full-test probability exactly matched the initial run, and the final CSV was byte-for-byte identical. Thus prior outputs are optional acceleration, not an extra data requirement. Generate the full candidate with `0.5 * leaf5.predict_proba(X)[:, 1] + 0.5 * leaf50.predict_proba(X)[:, 1]`, align to sample indices, and export index/target. Software versions are recorded in experiment_metadata.json. The separate genuine reproduction outputs and proof are also saved in the result directory.

Complete source and genuine outputs: `e1156285_lightgbm_leaf_blend.ipynb`. Results, checking scores, two component model files, the fixed experimental submission and independent verification: `lightgbm_leaf_blend_results/`.

**Student ownership and AI disclosure:** on 12 October 2026, the student explicitly chose the fixed 5/50 blend after reviewing the leaf-size results. OpenAI Codex implemented and evaluated that choice, verified artifacts and drafted this record. The student remains responsible for understanding, interpreting, selecting and submitting the model and accurately disclosing assistance. No further experiment, alternative weight, automatic final-model selection or Kaggle upload occurred.

### Every chronological result

| Train through | Check steps | Ensemble | Leaf 5 | Leaf 50 | Fixed blend |
|---:|---|---:|---:|---:|---:|
| 7 | 8–21 | 0.992985 | 0.991691 | 0.992588 | 0.992600 |
| 8 | 9–22 | 0.993827 | 0.993200 | 0.993597 | 0.993669 |
| 9 | 10–23 | 0.991915 | 0.990934 | 0.991773 | 0.991556 |
| 10 | 11–24 | 0.977749 | 0.976330 | 0.975622 | 0.976637 |
| 11 | 12–25 | 0.975593 | 0.973871 | 0.971731 | 0.973352 |
| 12 | 13–26 | 0.969104 | 0.968787 | 0.978432 | 0.975293 |
| 13 | 14–27 | 0.971758 | 0.969207 | 0.976040 | 0.974217 |
| 14 | 15–28 | 0.972177 | 0.968360 | 0.978591 | 0.975515 |
| 15 | 16–29 | 0.939024 | 0.943194 | 0.947238 | 0.945932 |
| 16 | 17–30 | 0.947183 | 0.948616 | 0.946034 | 0.947183 |
| 17 | 18–31 | 0.939401 | 0.941784 | 0.946074 | 0.943988 |
| 18 | 19–32 | 0.916818 | 0.923267 | 0.929097 | 0.925994 |
| 19 | 20–33 | 0.912879 | 0.918555 | 0.927562 | 0.923514 |
| 20 | 21–34 | 0.907205 | 0.908792 | 0.905247 | 0.907018 |
| 21 | 22–35 | 0.916956 | 0.919396 | 0.916709 | 0.917834 |
